In [24]:
import json
import re
from bisect import bisect_right
from typing import Iterable
# from .RelationE.sentences import sentence_spans

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
with open('../../data/entity_annotations_raw.jsonl', 'r', encoding='utf-8') as f:
    lines = [json.loads(line) for line in f.readlines()]

In [31]:
from dataclasses import dataclass, field
from typing import Any, Optional


@dataclass
class Mention:
    """One entity mention from the NER output. mention_id identifies the occurrence, not the real-world entity. Offsets refer to the bulletin text."""
    mention_id: str
    text: str
    entity_type: str
    start: int
    end: int
    normalized_name: Optional[str] = None
    confidence: Optional[float] = None
    sentence_id: Optional[int] = None          # assigned after sentence segmentation


_BOUNDARY = re.compile(r"(?<=[.!?])[\"')\]]*\s+(?=[A-Z0-9\"'(])")
_ABBREVIATIONS = frozenset("dr mr prof ir jhr st gen kol maj kapt luit adm mevr mej nr no blz enz bijv ca vgl".split())


def sentence_spans(text: str) -> list[tuple[int, int]]:
    """
    (start, end) of every sentence: a period/question/exclamation mark followed by
    space and a capital or digit, but not after an abbreviation or an initial.
    """
    cuts = {0, len(text)}
    for m in _BOUNDARY.finditer(text):
        before = re.search(r"(\w+)\.$", text[max(0, m.start() - 16):m.start()])
        if before and (before.group(1).lower() in _ABBREVIATIONS or (len(before.group(1)) == 1 and before.group(1).isupper())):
            continue
        cuts.add(m.end())
    points = sorted(cuts)
    spans = []
    for a, b in zip(points, points[1:]):
        seg = text[a:b]
        lead, trail = len(seg) - len(seg.lstrip()), len(seg) - len(seg.rstrip())
        if b - trail > a + lead:
            spans.append((a + lead, b - trail))
    return spans or [(0, len(text))]

def assign_sentence_ids(mentions: Iterable[Mention], spans: list[tuple[int, int]]) -> None:
    """Set mention.sentence_id: the sentence that contains the midpoint of the mention, otherwise the one with the largest overlap (None if the mention overlaps none)."""
    starts = [s for s, _ in spans]
    for m in mentions:
        mid = (m.start + m.end) // 2
        i = bisect_right(starts, mid) - 1
        if 0 <= i < len(spans) and spans[i][0] <= mid < spans[i][1]:
            m.sentence_id = i
            continue
        overlaps = [(max(0, min(m.end, e) - max(m.start, s)), -k) for k, (s, e) in enumerate(spans)]
        best = max(overlaps, default=(0, 0))
        m.sentence_id = -best[1] if best[0] > 0 else None

def get_character_distance(
    entity1: Mention,
    entity2: Mention,
) -> int:
    """
    Return the number of characters between two mention spans.

    Returns zero if spans overlap.
    """

    left, right = sorted(
        [entity1, entity2],
        key=lambda entity: entity.start,
    )

    return max(0, right.start - left.end)

In [23]:
record = lines[4]
txt = record['text']

s = sentence_spans(txt)
for (i, j) in s:
    print(txt[i:j])

RADIO 1 van 24 JULI 1939.
Goedenavond dames en heeren.
Nieuwsberichten van het ANP.
Publicatie, in welken vorm ook, is verbodn.
Eerste uitz. vvoor de NCRV (2e 10,-) Uit de Staatscourant: Aan den heer .C. VAN BEEK is op zijn verzoek met ingang van 7 AUGUSTUS 1939 eervol ontslag verleend als Burgemeester der gemeente DEURNE, met dank voor de door hem bewezen diensten.
Met ingang van 1 SEPTEMBER aanstaande is op zijn verzoek eervol ontslag verleend aan Dr. DE BRUYN als directeur van het Rijkslandbouwproefstation voor Veevoederonderzoek in WAGENINGEN, onder dankbetuiging voor de door hem bewezen diensten.
Het stoffelijk overschot van den beeldhouwer Dr. MENDES DA COSTA is vandaag op de begraafplaats van de Portugeesch Israelitische gemeente in OUDERKERK AAN DEN AMSTEL ten grave gedragen.
Onder de talrijke belangstellenden bevonden zich Prof. VAN MOERKERKEN van de Rijksacademie van beeldende kunsten in AMSTERDAM als vertegenwoordiger van den minister van Onderwijs, Kunsten en Wetenschappen,

In [35]:
m0 = Mention('1', 'RADIO 1', 'Organization', 0, 7)
m1 = Mention('2', 'ANP', 'Organization', 79, 82)


assign_sentence_ids([m0, m1], s)
m0, m1

(Mention(mention_id='1', text='RADIO 1', entity_type='Organization', start=0, end=7, normalized_name=None, confidence=None, sentence_id=0),
 Mention(mention_id='2', text='ANP', entity_type='Organization', start=79, end=82, normalized_name=None, confidence=None, sentence_id=2))

In [37]:
get_character_distance(m1, m0)

72

In [40]:
def is_within_sentence_distance(
    entity1: Mention,
    entity2: Mention,
    max_sentence_distance: int,
) -> bool:
    """
    Check sentence proximity.

    If either sentence assignment is unknown, return True and
    let character distance act as the fallback filter.
    """

    if (
        entity1.sentence_id is None
        or entity2.sentence_id is None
    ):
        return True

    distance = abs(
        entity1.sentence_id - entity2.sentence_id
    )

    return distance <= max_sentence_distance

def get_context_boundaries(
    entity1: Mention,
    entity2: Mention,
    text: str,
    sentence_spans: list[tuple[int, int]],
    padding: int,
) -> tuple[int, int]:
    """
    Extract a context window covering the sentences containing
    both mentions.

    If sentence assignments are unavailable, use the mention
    spans directly.
    """

    if (
        entity1.sentence_id is not None
        and entity2.sentence_id is not None
    ):

        first = min(
            entity1.sentence_id,
            entity2.sentence_id,
        )

        last = max(
            entity1.sentence_id,
            entity2.sentence_id,
        )

        start = sentence_spans[first][0]
        end = sentence_spans[last][1]

    else:

        start = min(entity1.start, entity2.start)
        end = max(entity1.end, entity2.end)

    start = max(0, start - padding)
    end = min(len(text), end + padding)

    return start, end


is_within_sentence_distance(m0, m1, 2)

True

In [46]:
pp = get_context_boundaries(m0, m1, txt, s, 0)

In [47]:
txt[pp[0]: pp[1]]

'RADIO 1 van 24 JULI 1939. Goedenavond dames en heeren. Nieuwsberichten van het ANP.'

In [51]:
import pandas
from candidate_generation import generate_candidate_pairs
from jev_ai import build_classification_input

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
# b = {"id": "anp:anp:1939:07:17:4:mpeg21", "created_at": "2026-09-09T12:27:11.496552Z", "updated_at": "2026-09-28T13:38:47.484786Z", "entities": [{"id": "gOMJtrnNAk", "text": "LONDEN", "start": 63, "end": 69, "type": "LOC", "qid": "Q84", "name_org": "LONDEN", "start_org": 65, "end_org": 71}, {"id": "RfYqXVg2h7", "text": "CHAMBERIAIN", "start": 159, "end": 170, "type": "PER", "qid": "Q10664", "name_org": "CHAMBERIAIN", "start_org": 162, "end_org": 173}, {"id": "R5a1obOidd", "text": "TOKIO", "start": 459, "end": 464, "type": "LOC", "qid": "Q1490", "name_org": "TOKIO", "start_org": 464, "end_org": 469}, {"id": "_gcFJTuD_h", "text": "Engeland", "start": 559, "end": 567, "type": "LOC", "qid": "Q21", "name_org": "Engeland", "start_org": 564, "end_org": 572}, {"id": "aQa3hHOxKn", "text": "FERNAND DE BRION", "start": 619, "end": 635, "type": "PER", "qid": None, "name_org": "FERNAND DE BRION", "start_org": 624, "end_org": 640}, {"id": "RgopoiNxdw", "text": "ARITA", "start": 498, "end": 503, "type": "PER", "qid": "Q668406", "name_org": "ARITA", "start_org": 503, "end_org": 508}, {"id": "SLYZLsxvmC", "text": "Rusland", "start": 302, "end": 309, "type": "LOC", "qid": "Q159", "name_org": "Rusland", "start_org": 305, "end_org": 312}, {"id": "LEj04ctggT", "text": "JERUZAIEM", "start": 1117, "end": 1126, "type": "LOC", "qid": "Q1218", "name_org": "JERUZAIEM", "start_org": 1126, "end_org": 1135}, {"id": "TatUzaSRTW", "text": "CARDINGTON", "start": 1054, "end": 1064, "type": "LOC", "qid": "Q7275217", "name_org": "CARDINGTON", "start_org": 1063, "end_org": 1073}, {"id": "doqw9zC_Zp", "text": "PARIJS", "start": 640, "end": 646, "type": "LOC", "qid": "Q90", "name_org": "PARIJS", "start_org": 645, "end_org": 651}, {"id": "nVT3FNWp2m", "text": "Iagerhuis", "start": 134, "end": 143, "type": "ORG", "qid": "Q11005", "name_org": "Iagerhuis", "start_org": 137, "end_org": 146}, {"id": "0NfHcQuw4g", "text": "comite Frankrijk-Duitschland", "start": 669, "end": 697, "type": "ORG", "qid": None, "name_org": "comite Frankrijk-Duitschland", "start_org": 675, "end_org": 703}, {"id": "UudPK5zHsP", "text": "joodsche nationale raad", "start": 1210, "end": 1233, "type": "ORG", "qid": "Q47932", "name_org": "joodsche nationale raad", "start_org": 1219, "end_org": 1242}], "text": "RADIO 2. Tweede uitzending van 17 Juli 1939. Copie BUITENIAND. LONDEN: In antwoord op een vraag van een arbeidersafgevaardigde in het Iagerhuis heeft minister CHAMBERIAIN verklaard, dat het hem niet mogelijk was de verzekering te geven, dat het Huis niet op reces zou gaan voor de onderhandelingen met Rusland geeindigd zouden zijn. Het is waarschijlijk dat vanavond de opmerkingen van de engelsche regeering over het rapport van den britschen ambassadeur in TOKIO over zijn onderhoud met minister ARITA, vanavond naar den diplomatieken vertegenwoordiger van Engeland in de japansche hoofdstad - zyllen worden geseind. FERNAND DE BRION HZZ PARIJS: De president van het comite Frankrijk-Duitschland, heeft een verklaring gepubliceerd waarin hij zegt: Niemand lan. meer naar een volledig onderzoek verlangen over het uitdeelen van buitenlandsche fondsen, dan TK, Ik wensch een onderzoek bij het bestuur van het comite omdat een dergelijk onderzoek een eind aan den laster zal maken. Vanmiddag zijn vijf ballons van een luchtversperring boven het vliegveld CARDINGTON door den bliksem getroffen en brandend neergestort. JERUZAIEM Uit protest tegen de voorgestelde schorsing van de joodsche im 5 migratie heeft de joodsche nationale raad een staking van tien uur geproclameerd, Van morgen 14 uur tot middernacht zal deze staking van arbeid, handel en verkeer duren.", "text_org": "RADIO 2. Tweede uitzending van I7 Juli I939. Copie BUITENIAND.\n1\nLONDEN : In antwoord op een vraag van een arbeidersafgevaardigde in het\nIagerhuis heeft minister CHAMBERIAIN verklaard, dat het hem niet mogelijk\nwas de verzekering te geven, dat het Huis niet op reces zou gaan voor de\nonderhandelingen met Rusland geeindigd zouden zijn.\n2\nHet is waarschijlijk dat vanavond de opmerkingen van de engelsche regeering\nover het rapport van den britschen ambassadeur in TOKIO over zijn onderhoud\nmet minister ARITA, vanavond naar den diplomatieken vertegenwoordiger van\nEngeland in de japansche hoofdstad - zyllen worden geseind.\nFERNAND DE BRION\nHZZ\nPARIJS : De president van het comite Frankrijk-Duitschland, heeft een verklaring gepubliceerd waarin hij zegt : Niemand lan. meer naar een volledig\nonderzoek verlangen over het uitdeelen van buitenlandsche fondsen, dan TK,\nIk wensch een onderzoek bij het bestuur van het comite omdat een dergelijk\nonderzoek een eind aan den laster zal maken.\n3\nVanmiddag zijn vijf ballons van een luchtversperring boven het vliegveld\nCARDINGTON door den bliksem getroffen en brandend neergestort.\nJERUZAIEM Uit protest tegen de voorgestelde schorsing van de joodsche im 5\nmigratie heeft de joodsche nationale raad een staking van tien uur geproclameerd, Van morgen I4 uur tot middernacht zal deze staking van arbeid,\nhandel en verkeer duren."}
# text = "RADIO 2. Tweede uitzending van I7 Juli I939. Copie BUITENIAND.\n1\nLONDEN : In antwoord op een vraag van een arbeidersafgevaardigde in het\nIagerhuis heeft minister CHAMBERIAIN verklaard, dat het hem niet mogelijk\nwas de verzekering te geven, dat het Huis niet op reces zou gaan voor de\nonderhandelingen met Rusland geeindigd zouden zijn.\n2\nHet is waarschijlijk dat vanavond de opmerkingen van de engelsche regeering\nover het rapport van den britschen ambassadeur in TOKIO over zijn onderhoud\nmet minister ARITA, vanavond naar den diplomatieken vertegenwoordiger van\nEngeland in de japansche hoofdstad - zyllen worden geseind.\nFERNAND DE BRION\nHZZ\nPARIJS : De president van het comite Frankrijk-Duitschland, heeft een verklaring gepubliceerd waarin hij zegt : Niemand lan. meer naar een volledig\nonderzoek verlangen over het uitdeelen van buitenlandsche fondsen, dan TK,\nIk wensch een onderzoek bij het bestuur van het comite omdat een dergelijk\nonderzoek een eind aan den laster zal maken.\n3\nVanmiddag zijn vijf ballons van een luchtversperring boven het vliegveld\nCARDINGTON door den bliksem getroffen en brandend neergestort.\nJERUZAIEM Uit protest tegen de voorgestelde schorsing van de joodsche im 5\nmigratie heeft de joodsche nationale raad een staking van tien uur geproclameerd, Van morgen I4 uur tot middernacht zal deze staking van arbeid,\nhandel en verkeer duren."
# entities = [{"id": "e1", "text": "LONDEN", "type": "Place", "start": 64, "end": 70}, {"id": "e2", "text": "Iagerhuis", "type": "Organization", "start": 136, "end": 145}, {"id": "e3", "text": "CHAMBERIAIN", "type": "Person", "start": 161, "end": 172}, {"id": "e4", "text": "Huis", "type": "Organization", "start": 246, "end": 250}, {"id": "e5", "text": "Rusland", "type": "Place", "start": 355, "end": 362}, {"id": "e6", "text": "TOKIO", "type": "Place", "start": 453, "end": 458}, {"id": "e7", "text": "ARITA", "type": "Person", "start": 493, "end": 498}, {"id": "e8", "text": "Engeland", "type": "Place", "start": 553, "end": 560}, {"id": "e9", "text": "FERNAND DE BRION", "type": "Person", "start": 610, "end": 616}, {"id": "e10", "text": "PARIJS", "type": "Place", "start": 630, "end": 636}, {"id": "e11", "text": "comite Frankrijk-Duitschland", "type": "Organization", "start": 660, "end": 687}, {"id": "e1", "text": "ballons", "type": "Artifact", "start": 1010, "end": 1017}, {"id": "e2", "text": "CARDINGTON", "type": "Place", "start": 1063, "end": 1073}, {"id": "e3", "text": "JERUZAIEM", "type": "Place", "start": 1126, "end": 1135}]
b = {"bulletin_id": "radio:1939:09:30:01", "bulletin_date": "1939-09-30", "text": "Vandaag ondertekende Chamberlain het Verdrag van Munchen na zijn onderhoud met Hitler, terwijl hij de Britsche Regeering leidt.", "entities": [{"id": "e1", "text": "Chamberlain", "start": 21, "end": 32, "type": "Person", "subtype": "Politician", "normalized_name": None, "annotation_confidence": 1.0}, {"id": "e2", "text": "Hitler", "start": 79, "end": 85, "type": "Person", "subtype": "Politician", "normalized_name": None, "annotation_confidence": 1.0}, {"id": "e3", "text": "Verdrag van Munchen", "start": 37, "end": 56, "type": "LegalDocument", "subtype": "Treaty", "normalized_name": None, "annotation_confidence": 1.0}, {"id": "e4", "text": "Britsche Regeering", "start": 102, "end": 120, "type": "Organization", "subtype": "Government", "normalized_name": None, "annotation_confidence": 1.0}], "gold_relations": [{"source": "e1", "type": "interacts_with", "target": "e2", "evidence": "onderhoud met Hitler"}, {"source": "e1", "type": "signed", "target": "e3", "evidence": "ondertekende Chamberlain het Verdrag van Munchen"}, {"source": "e1", "type": "leads", "target": "e4", "evidence": "hij de Britsche Regeering leidt"}]}


In [48]:
cand = generate_candidate_pairs(
    text=b['text'],
    entities=b['entities'],
    threshold = 250,
    context_pad = 80,
    # schema: dict | None = None,
    # max_pairs: int | None = None,
) 


In [50]:
cand[1]

{'pair_id': 'e1__e3',
 'source': 'e1',
 'source_type': 'Person',
 'target': 'e3',
 'target_type': 'LegalDocument',
 'candidate_types': ['introduces', 'signed'],
 'snippet': 'Vandaag ondertekende <e id="e1" type="Person">Chamberlain</e> het <e id="e3" type="LegalDocument">Verdrag van Munchen</e> na zijn onderhoud met Hitler, terwijl hij de Britsche Regeering leidt.',
 'snippet_start': 0,
 'snippet_end': 127,
 'distance': 5}

In [54]:
p=build_classification_input(bulletin=b, candidate_pairs=cand)
p

'{"bulletin_id": "radio:1939:09:30:01", "bulletin_date": "1939-09-30", "pairs": [{"pair_id": "e1__e2", "source": "e1", "source_type": "Person", "target": "e2", "target_type": "Person", "candidate_types": ["interacts_with"], "snippet": "Vandaag ondertekende <e id=\\"e1\\" type=\\"Person\\">Chamberlain</e> het Verdrag van Munchen na zijn onderhoud met <e id=\\"e2\\" type=\\"Person\\">Hitler</e>, terwijl hij de Britsche Regeering leidt."}, {"pair_id": "e1__e3", "source": "e1", "source_type": "Person", "target": "e3", "target_type": "LegalDocument", "candidate_types": ["introduces", "signed"], "snippet": "Vandaag ondertekende <e id=\\"e1\\" type=\\"Person\\">Chamberlain</e> het <e id=\\"e3\\" type=\\"LegalDocument\\">Verdrag van Munchen</e> na zijn onderhoud met Hitler, terwijl hij de Britsche Regeering leidt."}, {"pair_id": "e1__e4", "source": "e1", "source_type": "Person", "target": "e4", "target_type": "Organization", "candidate_types": ["affiliated_with", "interacts_with", "leads", "me

In [55]:
print(p)

{"bulletin_id": "radio:1939:09:30:01", "bulletin_date": "1939-09-30", "pairs": [{"pair_id": "e1__e2", "source": "e1", "source_type": "Person", "target": "e2", "target_type": "Person", "candidate_types": ["interacts_with"], "snippet": "Vandaag ondertekende <e id=\"e1\" type=\"Person\">Chamberlain</e> het Verdrag van Munchen na zijn onderhoud met <e id=\"e2\" type=\"Person\">Hitler</e>, terwijl hij de Britsche Regeering leidt."}, {"pair_id": "e1__e3", "source": "e1", "source_type": "Person", "target": "e3", "target_type": "LegalDocument", "candidate_types": ["introduces", "signed"], "snippet": "Vandaag ondertekende <e id=\"e1\" type=\"Person\">Chamberlain</e> het <e id=\"e3\" type=\"LegalDocument\">Verdrag van Munchen</e> na zijn onderhoud met Hitler, terwijl hij de Britsche Regeering leidt."}, {"pair_id": "e1__e4", "source": "e1", "source_type": "Person", "target": "e4", "target_type": "Organization", "candidate_types": ["affiliated_with", "interacts_with", "leads", "member_of"], "snipp

In [56]:
type(p)

str